# LLM Foundations to Agents — Intensive 3-Hour Lab
**Hardware:** NVIDIA T4 GPU (16 GB VRAM) &nbsp;|&nbsp; **Level:** Beginner–Intermediate

| # | Section | Mode | Time |
|---|---|---|---|
| 1 | Foundations | 👀 Run & Observe | 45 min |
| 2 | Quantization | ✏️ Fill config fields | 30 min |
| 3 | Inference & Prompting | ✏️ Set params & run | 30 min |
| 4 | RAG with FAISS | ✏️ Wire pipeline steps | 45 min |
| 5 | Agents with CrewAI | ✏️ Define agents & tasks | 30 min |

> **Convention:** Every blank you must fill is marked `# TODO`.


In [ ]:
# @title ⚙️ Setup — Run First (takes ~2 min)
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q faiss-gpu sentence-transformers
!pip install -q crewai crewai-tools
!pip install -q ctransformers huggingface_hub datasets
!pip install -q torch torchvision
print("\n✅ All packages installed.")
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — switch runtime to T4"
print(f"GPU: {gpu}")

---
## 👀 Section 1: Foundations — Run & Observe
**45 min | Tasks 1–5** — No blanks. Read each cell, run it, understand the output.


In [ ]:
# @title Task 1 — BPE: How tokenizers build their vocabulary
# BPE starts with characters and iteratively merges the most frequent adjacent pair.
from collections import defaultdict

def get_vocab(corpus):
    vocab = defaultdict(int)
    for word in corpus:
        vocab[' '.join(list(word)) + ' </w>'] += 1
    return dict(vocab)

def get_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        syms = word.split()
        for i in range(len(syms)-1):
            pairs[(syms[i], syms[i+1])] += freq
    return dict(pairs)

def merge_vocab(pair, vocab):
    import re
    bigram  = re.escape(' '.join(pair))
    pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    return {pattern.sub(''.join(pair), w): f for w, f in vocab.items()}

corpus = ['low','lower','newest','widest','low','low']
vocab  = get_vocab(corpus)
print("Initial vocab:", vocab)
for i in range(4):
    best = max(get_stats(vocab), key=get_stats(vocab).get)
    vocab = merge_vocab(best, vocab)
    print(f'Merge {i+1}: {"+".join(best)} -> {""}.join(best)}')
print("\nFinal vocab:", vocab)

In [ ]:
# @title Task 2 — Softmax: turning logits into probabilities
import numpy as np

def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)   # subtract max for numerical stability
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

logits = np.array([2.0, 1.0, 0.1, -1.0])
probs  = softmax(logits)
print(f"Logits: {logits}")
print(f"Probs:  {probs.round(4)}")
print(f"Sum:    {probs.sum():.6f}  <- must be 1.0")

In [ ]:
# @title Task 3 — Scaled Dot-Product Attention
import torch, math

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k    = Q.size(-1)
    scores = (Q @ K.transpose(-2,-1)) / math.sqrt(d_k)   # scale
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    attn = torch.softmax(scores, dim=-1)
    return attn @ V, attn

B, S, dk = 2, 5, 64
Q, K, V  = torch.randn(B,S,dk), torch.randn(B,S,dk), torch.randn(B,S,dk)
out, aw  = scaled_dot_product_attention(Q, K, V)
print(f"Output shape      : {out.shape}")
print(f"Attn weight shape : {aw.shape}")
print(f"Row-0 sum         : {aw[0,0].sum().item():.6f}  <- must be 1.0")

In [ ]:
# @title Task 4 — MoE: Top-1 Expert Routing
import torch, torch.nn as nn

class MoETop1Router(nn.Module):
    def __init__(self, d_model, num_experts):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts, bias=False)
    def forward(self, x):
        probs   = torch.softmax(self.gate(x), dim=-1)
        indices = probs.argmax(dim=-1)          # pick best expert per token
        return indices, probs

router = MoETop1Router(128, 8)
x      = torch.randn(2, 10, 128)  # batch=2, seq=10
ei, rp = router(x)
print(f"Expert chosen per token (batch 0): {ei[0].tolist()}")
print(f"Router probs shape               : {rp.shape}")
print(f"Probs sum to 1? {rp.sum(-1).allclose(torch.ones(2,10), atol=1e-4)}")

In [ ]:
# @title Task 5 — Transformer Forward Pass via HuggingFace (Demo)
# See the full pipeline in one call — tokenise → embed → attend → logits
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok   = AutoTokenizer.from_pretrained('facebook/opt-125m')
model = AutoModelForCausalLM.from_pretrained('facebook/opt-125m')
model.eval()

inputs = tok('The capital of France is', return_tensors='pt')
with torch.no_grad():
    out = model(**inputs)
next_token_id   = out.logits[0, -1].argmax().item()
next_token_text = tok.decode([next_token_id])
print(f"Logits shape : {out.logits.shape}  (batch, seq_len, vocab_size)")
print(f"Next token   : '{next_token_text}'")

In [ ]:
# @title 🔍 Section 1 Grader { display-mode: "form" }
import numpy as np, torch

_s1_passed = _s1_total = 0

def run_test(label, fn):
    global _s1_passed, _s1_total
    _s1_total += 1
    try:
        fn()
        print(f'  ✅ PASS  {label}')
        _s1_passed += 1
    except Exception as e:
        print(f'  ❌ FAIL  {label}\n         -> {e}')

print('Task 1 — get_stats')
def _t1a():
    s = get_stats({'a b </w>':3, 'a b c </w>':2})
    assert s.get(('a','b'),0)==5, f"expected 5, got {s.get(('a','b'),0)}"
def _t1b():
    s = get_stats({'x y </w>':4})
    assert s.get(('x','y'),0)==4 and s.get(('y','</w>'),0)==4
run_test('pair (a,b) count=5 across two vocab entries', _t1a)
run_test('single entry: all pairs inherit word frequency', _t1b)

print('Task 2 — softmax')
def _t2a():
    o = softmax(np.array([0.,0.,0.]))
    assert np.allclose(o,[1/3,1/3,1/3],atol=1e-5), f'got {o}'
def _t2b():
    o = softmax(np.array([1000.,1001.,1002.]))
    assert not np.any(np.isnan(o)), 'NaN — subtract max missing'
    assert o[2]>o[1]>o[0], 'ordering wrong'
run_test('uniform input -> [1/3, 1/3, 1/3]', _t2a)
run_test('large values: no NaN, ordering preserved', _t2b)

print('Task 3 — attention')
def _t3a():
    Q,K,V = torch.randn(2,4,16),torch.randn(2,4,16),torch.randn(2,4,16)
    o,aw = scaled_dot_product_attention(Q,K,V)
    assert tuple(o.shape)==(2,4,16), f'shape {o.shape}'
    assert torch.allclose(aw.sum(-1),torch.ones(2,4),atol=1e-4)
def _t3b():
    mask = torch.tensor([[[False,True],[False,False]]])
    Q,K,V = torch.randn(1,2,8),torch.randn(1,2,8),torch.randn(1,2,8)
    _,aw = scaled_dot_product_attention(Q,K,V,mask=mask)
    assert aw[0,0,1].item()<1e-6, f'masked weight={aw[0,0,1].item():.6f}'
run_test('output shape (2,4,16) and attn rows sum to 1', _t3a)
run_test('masked position receives ~0 attention weight', _t3b)

print('Task 4 — MoE router')
def _t4a():
    r = MoETop1Router(32,6); ei,rp = r(torch.randn(2,5,32))
    assert tuple(ei.shape)==(2,5) and tuple(rp.shape)==(2,5,6)
    assert torch.allclose(rp.sum(-1),torch.ones(2,5),atol=1e-4)
def _t4b():
    r = MoETop1Router(16,4); ei,rp = r(torch.randn(1,3,16))
    assert (ei>=0).all() and (ei<4).all(), f'index out of range: {ei}'
    assert torch.equal(ei, rp.argmax(-1)), 'indices must match argmax of probs'
run_test('shapes correct and router_probs sum to 1', _t4a)
run_test('indices in [0,num_experts) and match argmax', _t4b)

print(f"\nSection 1: {_s1_passed}/{_s1_total} tests passed")

---
## ✏️ Section 2: Quantization
**30 min | Tasks 6–10** — Configure BitsAndBytes. Fill every `# TODO`.


In [ ]:
# @title Task 6 — Memory Math: how much VRAM does a model need? [SOLUTION]

def compute_model_memory_gb(num_params_millions, dtype_bytes):
    return num_params_millions * 1e6 * dtype_bytes / 1e9

opt_fp32 = compute_model_memory_gb(125, 4)
opt_fp16 = compute_model_memory_gb(125, 2)
opt_int4 = compute_model_memory_gb(125, 0.5)
llm_fp16 = compute_model_memory_gb(7000, 2)
llm_int4 = compute_model_memory_gb(7000, 0.5)
print(f"OPT-125M  FP32={opt_fp32:.2f}GB  FP16={opt_fp16:.2f}GB  INT4={opt_int4:.3f}GB")
print(f"LLaMA-7B  FP16={llm_fp16:.1f}GB  INT4={llm_int4:.1f}GB")
print(f"7B INT4 fits in 16GB T4? {llm_int4 <= 16}")

In [ ]:
# @title Task 7 — Configure 4-bit NF4 Quantization [SOLUTION]
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)
print("BitsAndBytes config created:", bnb_config)

In [ ]:
# @title Task 8 — Load OPT-125M in 4-bit and measure VRAM [SOLUTION]
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'facebook/opt-125m'

torch.cuda.empty_cache()
vram_before = torch.cuda.memory_reserved(0) / 1e9

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto'
)

vram_after = torch.cuda.memory_reserved(0) / 1e9
print(f"VRAM before: {vram_before:.3f} GB")
print(f"VRAM after : {vram_after:.3f} GB")
print(f"Model used : {vram_after - vram_before:.3f} GB")

In [ ]:
# @title Task 9 — Enable Double Quantization [SOLUTION]
from transformers import BitsAndBytesConfig
import torch

bnb_config_dq = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
print(f"Standard NF4 double_quant: {bnb_config.bnb_4bit_use_double_quant}")
print(f"Double quant config      : {bnb_config_dq.bnb_4bit_use_double_quant}")

In [ ]:
# @title Task 10 — Inspect Quantized Layer Dtypes [SOLUTION]

for i, (name, param) in enumerate(model_4bit.named_parameters()):
    print(f"{name:<50} dtype={param.dtype}")
    if i >= 7:
        break

# uint8 = 4-bit weights packed into bytes by bitsandbytes

In [ ]:
# @title 🔍 Section 2 Grader { display-mode: "form" }
import torch
from transformers import BitsAndBytesConfig

_s2_passed = _s2_total = 0

def run_test(label, fn):
    global _s2_passed, _s2_total
    _s2_total += 1
    try:
        fn()
        print(f'  ✅ PASS  {label}')
        _s2_passed += 1
    except Exception as e:
        print(f'  ❌ FAIL  {label}\n         -> {e}')

print('Task 6 — memory math')
def _t6a():
    assert abs(compute_model_memory_gb(125,2)-0.25)<0.01, f'OPT-125M FP16 should be 0.25GB'
def _t6b():
    assert abs(compute_model_memory_gb(7000,0.5)-3.5)<0.1, f'7B INT4 should be ~3.5GB'
run_test('OPT-125M FP16 = 0.25 GB', _t6a)
run_test('LLaMA-7B INT4 = ~3.5 GB', _t6b)

print('Task 7 — bnb_config')
def _t7a():
    assert bnb_config is not None and isinstance(bnb_config, BitsAndBytesConfig)
    assert bnb_config.load_in_4bit == True
def _t7b():
    assert bnb_config.bnb_4bit_quant_type == 'nf4', f'got {bnb_config.bnb_4bit_quant_type}'
    assert bnb_config.bnb_4bit_compute_dtype == torch.float16
run_test('load_in_4bit=True and correct type', _t7a)
run_test('quant_type=nf4, compute_dtype=float16', _t7b)

print('Task 8 — model_4bit loaded')
def _t8a():
    assert model_4bit is not None, 'model_4bit is None'
def _t8b():
    dtypes = {p.dtype for _,p in model_4bit.named_parameters()}
    assert torch.uint8 in dtypes, f'No uint8 params found — model may not be 4-bit. Dtypes: {dtypes}'
run_test('model_4bit is not None', _t8a)
run_test('at least one uint8 (4-bit) layer present', _t8b)

print('Task 9 — double quant config')
def _t9a():
    assert bnb_config_dq is not None and bnb_config_dq.bnb_4bit_use_double_quant == True
def _t9b():
    assert bnb_config_dq.bnb_4bit_quant_type == 'nf4'
    assert bnb_config.bnb_4bit_use_double_quant == False, 'original config must stay False'
run_test('double_quant=True in new config', _t9a)
run_test('quant_type still nf4, original unchanged', _t9b)

print(f"\nSection 2: {_s2_passed}/{_s2_total} tests passed")

---
## ✏️ Section 3: Inference & Prompting
**30 min | Tasks 11–15** — Load a model, tune decoding parameters, and engineer prompts.


In [ ]:
# @title Task 11 — Load TinyLlama GGUF via ctransformers [SOLUTION]
from huggingface_hub import hf_hub_download
from ctransformers import AutoModelForCausalLM as CT_Model

REPO = 'TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF'
FILE = 'tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf'

gguf_path  = hf_hub_download(repo_id=REPO, filename=FILE)
gguf_model = CT_Model.from_pretrained(
    gguf_path,
    model_type='llama',
    gpu_layers=50
)

out = gguf_model('Hello!', max_new_tokens=10)
print(f"Model loaded. Quick test: {out[:80]}")

In [ ]:
# @title Task 12 — Explore Temperature: how it changes response style [SOLUTION]

PROMPT = '<|system|>\nYou are a helpful assistant.\n<|user|>\nDescribe the ocean in one sentence.\n<|assistant|>\n'

# Three temperatures: low / mid / high
temperatures_to_try = [0.2, 0.8, 1.5]

for temp in temperatures_to_try:
    response = gguf_model(PROMPT, max_new_tokens=60, temperature=temp, repetition_penalty=1.1)
    print(f'--- temperature={temp} ---')
    print(response.strip()[:200])
    print()

In [ ]:
# @title Task 13 — Explore Top-P and Top-K Sampling [SOLUTION]

PROMPT2 = '<|system|>\nYou are a creative storyteller.\n<|user|>\nBegin a story: The robot opened its eyes and\n<|assistant|>\n'

experiments = [
    {'label': 'wide nucleus',   'top_p': 0.95, 'top_k': 0 },
    {'label': 'narrow nucleus', 'top_p': 0.3,  'top_k': 0 },
    {'label': 'top-k only',     'top_p': 1.0,  'top_k': 10},
]

for exp in experiments:
    response = gguf_model(
        PROMPT2, max_new_tokens=80, temperature=0.8,
        top_p=exp['top_p'], top_k=exp['top_k'],
        repetition_penalty=1.1
    )
    print(f'--- {exp["label"]} (top_p={exp["top_p"]}, top_k={exp["top_k"]}) ---')
    print(response.strip()[:250])
    print()

In [ ]:
# @title Task 14 — Few-Shot Prompt Engineering [SOLUTION]

few_shot_prompt = """Classify the sentiment: Positive or Negative.

Input: This product exceeded all my expectations!
Output: Positive

Input: Terrible quality, broke after one day.
Output: Negative

Input: Exactly what I needed, arrived quickly.
Output: Positive

Input: The packaging was damaged and the item was broken.
Output:"""

response = gguf_model(few_shot_prompt, max_new_tokens=5, temperature=0.1)
print("Prompt sent:")
print(few_shot_prompt)
print(f"\nModel answer: {response.strip()}")

In [ ]:
# @title Task 15 — Chain-of-Thought Prompting [SOLUTION]

QUESTION = 'A train travels 120 km in 1.5 hours, stops for 20 minutes, then covers 90 km in 1 hour. What is the average speed for the entire journey?'

cot_prompt = f"""<|system|>
You are a precise reasoning assistant.
<|user|>
{QUESTION}
Let's think step by step.
<|assistant|>
"""

response = gguf_model(cot_prompt, max_new_tokens=200, temperature=0.3)
print(response)

In [ ]:
# @title 🔍 Section 3 Grader { display-mode: "form" }
_s3_passed = _s3_total = 0

def run_test(label, fn):
    global _s3_passed, _s3_total
    _s3_total += 1
    try:
        fn()
        print(f'  ✅ PASS  {label}')
        _s3_passed += 1
    except Exception as e:
        print(f'  ❌ FAIL  {label}\n         -> {e}')

print('Task 11 — GGUF model')
run_test('gguf_model is loaded', lambda: (_ for _ in ()).throw(AssertionError('gguf_model is None')) if gguf_model is None else None)
def _t11b():
    out = gguf_model('Test', max_new_tokens=3)
    assert isinstance(out, str) and len(out)>0
run_test('model generates non-empty string output', _t11b)

print('Task 12 — temperature exploration')
def _t12a():
    assert len(temperatures_to_try)>=2, f'need >=2 temps, got {len(temperatures_to_try)}'
def _t12b():
    assert all(isinstance(t,(int,float)) and t>0 for t in temperatures_to_try), 'all temps must be positive numbers'
    assert max(temperatures_to_try)!=min(temperatures_to_try), 'try distinct temperature values'
run_test('at least 2 temperature values defined', _t12a)
run_test('all values positive and distinct', _t12b)

print('Task 13 — top-p / top-k exploration')
def _t13a():
    assert len(experiments)>=2, f'define >=2 experiments, got {len(experiments)}'
def _t13b():
    for e in experiments:
        assert 'top_p' in e and 'top_k' in e, f'experiment missing top_p or top_k: {e}'
run_test('at least 2 sampling experiments defined', _t13a)
run_test('each experiment has top_p and top_k keys', _t13b)

print('Task 14 — few-shot prompt')
def _t14a():
    assert few_shot_prompt.count('Input:')>=3, f'need >=3 Input: lines, got {few_shot_prompt.count("Input:")}'
def _t14b():
    assert few_shot_prompt.count('Output:')>=3
    assert few_shot_prompt.strip().endswith('Output:'), 'prompt must end with Output: for the new query'
run_test('3+ examples: 3 Input: lines present', _t14a)
run_test('3+ Output: lines, last one is open-ended', _t14b)

print('Task 15 — CoT prompt')
def _t15a():
    assert '<|system|>' in cot_prompt and '<|user|>' in cot_prompt and '<|assistant|>' in cot_prompt
def _t15b():
    assert 'step by step' in cot_prompt.lower(), 'CoT trigger phrase missing'
    assert QUESTION[:20] in cot_prompt, 'question not in prompt'
run_test('ChatML tags present', _t15a)
run_test('CoT trigger phrase and question in prompt', _t15b)

print(f"\nSection 3: {_s3_passed}/{_s3_total} tests passed")

---
## ✏️ Section 4: RAG with FAISS
**45 min | Tasks 16–20** — Build a full Retrieval-Augmented Generation pipeline.


In [ ]:
# @title Task 16 — Load Web Documents with LangChain [SOLUTION]
from langchain_community.document_loaders import WebBaseLoader

URLS = [
    'https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)',
    'https://en.wikipedia.org/wiki/Large_language_model',
]

docs = WebBaseLoader(URLS).load()

print(f"Documents loaded: {len(docs)}")
print(f"First doc length: {len(docs[0].page_content)} chars")

In [ ]:
# @title Task 17 — Chunk Documents with RecursiveCharacterTextSplitter [SOLUTION]
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64)
chunks   = splitter.split_documents(docs)

print(f"Chunks produced : {len(chunks)}")
print(f"Sample chunk    : {chunks[0].page_content[:150]}")

In [ ]:
# @title Task 18 — Create Sentence Embeddings [SOLUTION]
from langchain_community.embeddings import HuggingFaceEmbeddings

EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

vec = embeddings.embed_query('What is attention?')
print(f"Embedding dim: {len(vec)}")

In [ ]:
# @title Task 19 — Build FAISS Vector Index [SOLUTION]
from langchain_community.vectorstores import FAISS

print('Building FAISS index...')
vector_db = FAISS.from_documents(chunks, embeddings)

results = vector_db.similarity_search('How does attention mechanism work?', k=2)
print(f"Vectors indexed: {vector_db.index.ntotal}")
print(f"Top result snippet: {results[0].page_content[:150]}")

In [ ]:
# @title Task 20 — Assemble RetrievalQA Chain [SOLUTION]
from langchain.chains import RetrievalQA
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, pipeline

tok_opt = AutoTokenizer.from_pretrained('facebook/opt-125m')
hf_pipe = pipeline('text-generation', model=model_4bit, tokenizer=tok_opt,
                    max_new_tokens=128, do_sample=False)
llm     = HuggingFacePipeline(pipeline=hf_pipe)

retriever = vector_db.as_retriever(search_kwargs={'k': 3})
qa_chain  = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=True
)

result = qa_chain.invoke({'query': 'What is the Transformer architecture?'})
print(f"Answer: {result['result'][:300]}")
print(f"Sources: {len(result['source_documents'])}")

In [ ]:
# @title 🔍 Section 4 Grader { display-mode: "form" }
from langchain_community.vectorstores import FAISS as _FAISS
from langchain.chains import RetrievalQA as _RQA

_s4_passed = _s4_total = 0

def run_test(label, fn):
    global _s4_passed, _s4_total
    _s4_total += 1
    try:
        fn()
        print(f'  ✅ PASS  {label}')
        _s4_passed += 1
    except Exception as e:
        print(f'  ❌ FAIL  {label}\n         -> {e}')

print('Task 16 — document loading')
run_test('docs is a list with 2+ entries', lambda: (_ for _ in ()).throw(AssertionError()) if not (isinstance(docs,list) and len(docs)>=2) else None)
def _t16b():
    assert hasattr(docs[0],'page_content') and len(docs[0].page_content)>500
run_test('documents have substantial page_content', _t16b)

print('Task 17 — chunking')
def _t17a():
    assert isinstance(chunks,list) and len(chunks)>len(docs)
def _t17b():
    assert all(len(c.page_content)<=700 for c in chunks), 'some chunks too long'
run_test('chunks > docs (splitting happened)', _t17a)
run_test('all chunks within size limit', _t17b)

print('Task 18 — embeddings')
def _t18a():
    assert embeddings is not None
    v = embeddings.embed_query('test')
    assert len(v)==384, f'expected 384-dim, got {len(v)}'
def _t18b():
    import numpy as np
    v = embeddings.embed_query('hello')
    assert abs(np.linalg.norm(v)-1.0)<0.01, 'vectors not normalized'
run_test('embeddings produce 384-dim vectors', _t18a)
run_test('vectors are L2-normalized', _t18b)

print('Task 19 — FAISS index')
def _t19a():
    assert isinstance(vector_db,_FAISS) and vector_db.index.ntotal==len(chunks)
def _t19b():
    r = vector_db.similarity_search('attention', k=1)
    assert len(r)==1 and len(r[0].page_content)>10
run_test(f'index contains {len(chunks)} vectors', _t19a)
run_test('similarity_search returns relevant result', _t19b)

print('Task 20 — RetrievalQA chain')
def _t20a():
    assert isinstance(qa_chain,_RQA)
def _t20b():
    assert result is not None and 'result' in result and 'source_documents' in result
    assert len(result['result'])>5
run_test('qa_chain is a RetrievalQA instance', _t20a)
run_test('invoke returns result + source_documents', _t20b)

print(f"\nSection 4: {_s4_passed}/{_s4_total} tests passed")

---
## ✏️ Section 5: Agents with CrewAI
**30 min | Tasks 21–25** — Define agents, give them roles and tools, run a multi-agent crew.

CrewAI builds on top of LangChain. An `Agent` has a **role**, **goal**, and **backstory**.
A `Task` tells an agent what to produce. A `Crew` orchestrates multiple agents.


In [ ]:
# @title Task 21 — Configure the LLM backend for CrewAI [SOLUTION]
from langchain_community.llms import CTransformers
from crewai import Agent, Task, Crew, Process

crew_llm = CTransformers(
    model=gguf_path,
    model_type='llama',
    config={'max_new_tokens': 256, 'temperature': 0.3, 'gpu_layers': 50}
)

out = crew_llm('Hello in one word:')
print(f"LLM test: {out.strip()[:80]}")

In [ ]:
# @title Task 22 — Define a Researcher Agent [SOLUTION]

researcher = Agent(
    role='AI Research Analyst',
    goal='Summarise key facts about a given AI topic clearly and concisely',
    backstory='You are a senior ML researcher with 10 years of experience reading and distilling academic papers and technical blogs into clear bullet points.',
    llm=crew_llm,
    verbose=True,
    allow_delegation=False
)

print(f"Agent role: {researcher.role}")

In [ ]:
# @title Task 23 — Define a Writer Agent [SOLUTION]

writer = Agent(
    role='Technical Writer',
    goal='Turn research notes into a clear, engaging paragraph for a general audience',
    backstory='You are an experienced science communicator who excels at making complex AI concepts accessible without losing accuracy.',
    llm=crew_llm,
    verbose=True,
    allow_delegation=False
)

print(f"Agent role: {writer.role}")

In [ ]:
# @title Task 24 — Define Tasks for Each Agent [SOLUTION]

TOPIC = 'Retrieval-Augmented Generation (RAG)'

research_task = Task(
    description=f'Research the following topic and list 5 key facts: {TOPIC}',
    expected_output='A bullet-point list of 5 factual statements',
    agent=researcher
)

write_task = Task(
    description='Using the research notes, write a 3-sentence summary suitable for a blog post',
    expected_output='A 3-sentence paragraph',
    agent=writer
)

print(f"Tasks defined: research={research_task is not None}, write={write_task is not None}")

In [ ]:
# @title Task 25 — Assemble and Run the Crew [SOLUTION]

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=True
)

print('Running crew... (may take 1-2 min)')
crew_result = crew.kickoff()
print("\n===== CREW OUTPUT =====")
print(crew_result)

In [ ]:
# @title 🔍 Section 5 Grader { display-mode: "form" }
from crewai import Agent as _Agent, Task as _Task, Crew as _Crew

_s5_passed = _s5_total = 0

def run_test(label, fn):
    global _s5_passed, _s5_total
    _s5_total += 1
    try:
        fn()
        print(f'  ✅ PASS  {label}')
        _s5_passed += 1
    except Exception as e:
        print(f'  ❌ FAIL  {label}\n         -> {e}')

print('Task 21 — crew_llm')
def _t21a():
    assert crew_llm is not None
def _t21b():
    out = crew_llm('Say yes:')
    assert isinstance(out,str) and len(out)>0
run_test('crew_llm is not None', _t21a)
run_test('crew_llm generates string output', _t21b)

print('Task 22 — researcher agent')
def _t22a():
    assert isinstance(researcher,_Agent), f'got {type(researcher)}'
def _t22b():
    assert researcher.role and len(researcher.role)>3
    assert researcher.goal and len(researcher.goal)>10
    assert researcher.backstory and len(researcher.backstory)>10
run_test('researcher is a CrewAI Agent', _t22a)
run_test('role, goal, backstory all populated', _t22b)

print('Task 23 — writer agent')
def _t23a():
    assert isinstance(writer,_Agent)
def _t23b():
    assert writer.role != researcher.role, 'writer and researcher must have different roles'
run_test('writer is a CrewAI Agent', _t23a)
run_test('writer has a distinct role from researcher', _t23b)

print('Task 24 — tasks')
def _t24a():
    assert isinstance(research_task,_Task) and isinstance(write_task,_Task)
def _t24b():
    assert research_task.agent == researcher
    assert write_task.agent == writer
run_test('both tasks are CrewAI Task instances', _t24a)
run_test('tasks assigned to correct agents', _t24b)

print('Task 25 — crew execution')
def _t25a():
    assert isinstance(crew,_Crew) and len(crew.agents)==2
def _t25b():
    assert crew_result is not None and len(str(crew_result))>20
run_test('crew has 2 agents and 2 tasks', _t25a)
run_test('crew_result is non-empty', _t25b)

print(f"\nSection 5: {_s5_passed}/{_s5_total} tests passed")

---
## 🏆 Final Grader
Run this cell after completing all sections to see your overall score.


In [ ]:
# @title 🏆 Final Cumulative Grader { display-mode: "form" }
print('=' * 50)
print('       FINAL LAB SCORE REPORT')
print('=' * 50)

sections = [
    ('Section 1 — Foundations',       _s1_passed, _s1_total),
    ('Section 2 — Quantization',      _s2_passed, _s2_total),
    ('Section 3 — Inference',         _s3_passed, _s3_total),
    ('Section 4 — RAG with FAISS',    _s4_passed, _s4_total),
    ('Section 5 — Agents (CrewAI)',   _s5_passed, _s5_total),
]

grand_passed = grand_total = 0
for name, p, t in sections:
    bar   = '█' * p + '░' * (t - p)
    grand_passed += p
    grand_total  += t
    status = '✅' if p == t else ('⚠️ ' if p >= t//2 else '❌')
    print(f'  {status}  {name:<35} {p}/{t}  [{bar}]')

pct = int(grand_passed / grand_total * 100) if grand_total else 0
print('=' * 50)
print(f'  TOTAL: {grand_passed}/{grand_total} tests passed  ({pct}%)')
print('=' * 50)

if pct == 100:
    print('  🎉 Perfect score! All tasks complete.')
elif pct >= 80:
    print('  👍 Great work! Review any failing sections above.')
elif pct >= 50:
    print('  📖 Good progress. Revisit failing tasks before moving on.')
else:
    print('  🔧 Keep going — run section graders first, then rerun this cell.')

---
## What You Built

| Section | Packages used | Key concept |
|---|---|---|
| 1 Foundations | `numpy`, `torch`, `transformers` | BPE, softmax, attention, MoE |
| 2 Quantization | `bitsandbytes`, `transformers` | NF4 4-bit, VRAM budgeting |
| 3 Inference | `ctransformers` | Temperature, top-p, top-k, prompting |
| 4 RAG | `langchain`, `faiss-gpu`, `sentence-transformers` | Embed → Index → Retrieve → Generate |
| 5 Agents | `crewai` | Multi-agent roles, tasks, sequential crew |
